# 条件付き敵対的生成ネットワーク（cGAN: Conditional Generative Adversarial Networks）

**学習目標**

- Keras で Conditional GAN アーキテクチャ（Generator と Discriminator で構成）を構築する
- Generator と Discriminator の損失を定義する
- keras.Model のサブクラス化を使用して GAN のトレーニング ステップを定義する
- MNIST データセットで cGAN をトレーニングする

## はじめに
このノートブックでは、[条件付き敵対的生成ネットワーク（Conditional Generative Adversarial Network: cGAN）](https://arxiv.org/abs/1411.1784) の構築とトレーニングの方法を説明します。

前のラボ（[deepconv-gan.ipynb](./deepconv_gan.ipynb)）で説明した基本的な GAN は、生成方法を学習したい画像に対して完全に教師なしの方法でトレーニングされます。<br>
ランダムなノイズ ベクトルなどの潜在表現が、学習された画像空間を探索およびサンプリングするために使用されます。

単純な拡張として、ラベルを外部フラグとして入力に追加する方法があります。<br>
たとえば、0 から 9 までの手書き数字で構成される MNIST データセットを考えてみましょう。

通常、GAN は数字の分布を学習するだけであり、Generator にランダムなノイズ ベクトルが与えられると、さまざまな数字が生成されます。

![Basic GAN](https://user-images.githubusercontent.com/6895245/171208971-3ff1c023-cdd7-47c2-b734-77e07b8eb6ac.png)

しかし、どの数字が生成されるかを制御することはできません。

トレーニング中、MNIST のように各画像の実際の数字やクラス指定が分かっている場合があります。<br>
その追加情報を GAN トレーニングの特徴量として含めることができ、推論時に使用できます。

条件付き GAN（cGAN）を使用すると、画像生成を数字の条件付きにできるため、関心のある特定の数字の分布に焦点を絞ることができます。

そして、推論時にはランダムな数字を受け取る代わりに、希望する数字を渡すことで特定の数字の画像を作成できます。

![cGAN](https://user-images.githubusercontent.com/6895245/171209451-4b95c856-fafa-40b4-87b6-c7232c1de841.png)

このノートブックでは、条件付き GAN の構築方法を学びます。

## ライブラリのインポート

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [ ]:
import time

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from keras.layers import (
    BatchNormalization,
    Concatenate,
    Conv2D,
    Conv2DTranspose,
    Dense,
    Embedding,
    Flatten,
    Input,
    LeakyReLU,
    Reshape,
)

## 定数

In [ ]:
IMAGE_SHAPE = (28, 28, 1)
NUM_CLASSES = 10
LATENT_DIM = 512
PROJECTION_SHAPE = (7, 7, 56)

## Conditional GAN 用の Generator および Discriminator モデルの作成

前のラボでは、通常の GAN は [Keras Sequential API](https://keras.io/api/models/sequential/) を使用して定義されましたが、ここでは [Keras Functional API](https://keras.io/guides/functional_api/) を使用して Conditional GAN 用のより複雑なモデルを作成します。

まず、Generator と Discriminator の両方で使用される、数字（0〜9）情報を埋め込むためのレイヤを定義する必要があります。<br>
埋め込みは、後で MNIST 画像と連結できるように、画像のような rank 3 のテンソル（height, width, 1）にする必要があります。

**演習**: 以下を追加して `create_digit_embedding` 関数を完成させてください。
- クラス インデックスを `embedding_dim` 次元に埋め込むための `Embedding` レイヤ。
- 埋め込みの次元数を num_pixels（画像の height * width）次元に変更するための `Dense` レイヤ。
- 後で画像と連結するために、Dense レイヤの出力を 1D テンソルから 2D テンソルに変形するための `Reshape` レイヤ。

In [ ]:
def create_digit_embedding(digits, num_classes, embedding_dim, image_shape):
    # TODO: Embedding レイヤを定義する。
    embedded_digits = Embedding(...)(digits)

    num_pixels = image_shape[0] * image_shape[1]  # height * width

    # TODO: Dense レイヤを定義する。
    dense_digits = Dense(...)(embedded_digits)

    # TODO: Reshape レイヤを定義する
    digits_image = Reshape(...)(dense_digits)

    return digits_image

数字の埋め込み画像を生成してみましょう。まだトレーニングされていないため、単なるランダムな画像です。

In [ ]:
DIGIT_TO_EMBED = 0  # [0,9]

digit_embedding = create_digit_embedding(
    keras.ops.convert_to_tensor([DIGIT_TO_EMBED]), NUM_CLASSES, 128, (7, 7, 1)
)[0]
plt.imshow(digit_embedding)

### Generator

Conditional GAN の Generator は、ランダムな潜在ベクトルと数字ベクトルから画像を作成します。
`latent_dim` の shape のノイズ ベクトルと、数字のインデックスを受け取ります。両方のベクトルは埋め込まれ、連結されるように `projection_shape` で定義された同じ height と width に変形されます。その後、Generator は連結された rank 3 のベクトルから 28x28x1 の MNIST 画像を生成します。

Generator モデルは、通常 [BatchNormalization](https://keras.io/api/layers/normalization_layers/batch_normalization/) とともに画像をアップサンプリングするために [Conv2DTranspose](https://keras.io/api/layers/convolution_layers/conv2d_transpose/) を使用します。また、非線形活性化関数として [LeakyReLU](https://keras.io/api/layers/activation_layers/leaky_relu/) を使用します。

次に、cGAN Generator を構築するための関数 `create_dcgan_generator` を定義します。これには次の処理が含まれます。

1. 潜在ベクトルを rank 3 テンソルにマッピングします。ここでは、潜在画像の shape を定義するために `projection_shape` 変数を使用します。
2. 数字インデックスを rank 3 テンソルにマッピングします（`create_digit_embedding` 関数を呼び出します）。出力テンソルの shape は、次のステップで画像と連結されるように、`projection_shape` と同じ height および width にフォーマットされます。
3. 1 と 2 を連結します。
4. 3 を、`Conv2DTranspose`、`BatchNormalization`、`LeakyReLU` レイヤで構成されるブロックを持つ GAN Generator に渡します。この部分は前のノートブックの DCGAN Generator とほぼ同じであることに注目してください。
5. Model オブジェクトを返します。

**演習**: `create_dcgan_generator` 関数を完成させてください。
- 上記で定義した `create_digit_embedding` 関数を完成させます。
- `latent_image` と `digit_embeddings` を積み重ねるための Concatenate レイヤを追加します。

In [ ]:
def create_dcgan_generator(latent_dim, num_classes, projection_shape):
    # ランダム ベクトルの入力
    latent_vector = Input(name="latent_input", shape=(latent_dim,))

    num_image_pixel = (
        projection_shape[0] * projection_shape[1] * projection_shape[2]
    )

    x = Dense(name="latent_dense", units=num_image_pixel)(latent_vector)
    x = LeakyReLU(name="latent_leakyrelu", alpha=0.2)(x)

    latent_image = Reshape(
        name="latent_to_image_reshape", target_shape=projection_shape
    )(x)

    # 数字入力を受け取る ([0,9])
    digits = Input(name="digit_input", shape=())

    # TODO: create_digit_embedding 関数を完成させる
    # 数字を rank 3 テンソルの数字埋め込みに射影する。
    digit_embeddings = create_digit_embedding(...)

    # TODO: ここで画像とラベルを連結する。
    concatenated_inputs = Concatenate(...)(...)

    x = Conv2DTranspose(128, 4, 2, "same")(concatenated_inputs)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    x = Conv2DTranspose(128, 4, 2, "same")(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)

    outputs = Conv2DTranspose(
        1, 3, 1, "same", activation="tanh", name="generator_output"
    )(x)

    return keras.Model(
        inputs=[latent_vector, digits], outputs=outputs, name="dcgan_generator"
    )

In [ ]:
dcgan_generator = create_dcgan_generator(
    latent_dim=LATENT_DIM,
    num_classes=NUM_CLASSES,
    projection_shape=PROJECTION_SHAPE,
)
dcgan_generator.summary()

Generator モデルをプロットして、どのように構築されているか確認してみましょう。

In [ ]:
# Generator モデルをプロット
keras.utils.plot_model(
    model=dcgan_generator, show_shapes=True, show_layer_names=True, dpi=75
)

### Discriminator

次に、Discriminator を構築します。Discriminator は CNN ベースの画像分類器です。<br>
shape が 28x28x1 の画像と、同様に埋め込まれて 28x28x1 に変形された数字インデックスを受け取り、それらを連結して、その画像が本物か偽物かを示す単一の分類結果を返します。

cGAN Discriminator を構築するための関数 `create_dcgan_discriminator` を定義しましょう。これには次の処理が含まれます。

1. 本物または偽物のいずれかである、`image_shape` の shape の画像入力を受け取ります。
2. 数字インデックスを rank 3 テンソルにマッピングします（`create_digit_embedding` 関数を呼び出します）。出力テンソルの shape は、次のステップで画像と連結されるように、`image_shape` と同じ height および width にフォーマットされます。
3. 1 と 2 を連結します。
4. 3 を、`Conv2D`、`BatchNormalization`、`LeakyReLU` レイヤで構成されるブロックを持つ GAN Discriminator に渡します。この部分は前のノートブックの DCGAN Discriminator とほぼ同じであることに注目してください。
5. Model オブジェクトを返します。

**演習**: 上記の手順に従って、`create_dcgan_discriminator` 関数全体を完成させてください。

In [ ]:
def create_dcgan_discriminator(image_shape, num_classes):
    # TODO: 関数を完成させる。

    return keras.Model(
        inputs=[images, digits], outputs=outputs, name="dcgan_discriminator"
    )

In [ ]:
dcgan_discriminator = create_dcgan_discriminator(
    image_shape=IMAGE_SHAPE, num_classes=NUM_CLASSES
)
dcgan_discriminator.summary()

In [ ]:
# Discriminator モデルをプロット
keras.utils.plot_model(
    model=dcgan_discriminator, show_shapes=True, show_layer_names=True, dpi=75
)

Generator と Discriminator ネットワークのトレーニングを容易にするために使用できる、`GAN` という名前のカスタム `Model` クラスを作成します。

## トレーニング ループの定義

### Discriminator のトレーニング ステップ

Discriminator のトレーニング ステップを定義します。これには次のステップが含まれます。
1. Generator を使用して偽の画像を生成します。
2. 生成された偽の画像を本物の画像と結合します。
3. 結合された画像に対応する偽（0）と本物（1）のラベルを作成します。
4. Discriminator のトレーニングを支援するために、本物のラベルにラベル スムージング（ワンサイド ラベル スムージング）を適用します。
5. `tf.GradientTape` コンテキスト内で、結合された画像と数字インデックスを Discriminator に渡し、Discriminator の損失を計算します。
6. `GradientTape` から Discriminator のパラメータの勾配を取得し、それらを更新します。Generator は Discriminator のトレーニング ステップでは更新されないことに注意してください。

**演習**: Discriminator のトレーニング ステップを完成させてください。

In [ ]:
def discriminator_train_step(
    discriminator,
    generator,
    discriminator_optimizer,
    loss_fn,
    discriminator_accuracy,
    one_sided_label_smoothing,
    batch_size,
    real_images,
    input_digits,
):
    # 潜在空間内のランダムな点をサンプリングする。
    random_latent_vectors = keras.random.normal(shape=(batch_size, LATENT_DIM))

    # 生成された偽の画像をデコードする。
    generated_images = generator(
        inputs=[random_latent_vectors, input_digits], training=False
    )

    # 偽の画像と本物の画像を結合する。
    combined_images = keras.ops.concat([generated_images, real_images], axis=0)

    # 偽と本物のラベルを作成する。
    # TODO: ラベルを定義する。
    fake_labels = ...
    real_labels = ...

    # トレーニングを助けるために本物のラベルをスムージングする。
    # TODO: ワンサイド スムージングを適用する。
    real_labels = ...

    # 結合された画像に合わせてラベルを結合する。
    labels = keras.ops.concat([fake_labels, real_labels], axis=0)

    # Discriminator のみをトレーニングする。
    with tf.GradientTape() as tape:
        predictions = discriminator(
            inputs=[
                combined_images,
                keras.ops.concat([input_digits, input_digits], axis=0),
            ]
        )
        discriminator_loss = loss_fn(y_true=labels, y_pred=predictions)

    pred_prob = keras.ops.sigmoid(predictions)
    discriminator_accuracy.update_state(y_true=labels, y_pred=pred_prob)

    grads = tape.gradient(
        target=discriminator_loss,
        sources=discriminator.trainable_weights,
    )

    discriminator_optimizer.apply_gradients(
        grads_and_vars=zip(grads, discriminator.trainable_weights)
    )

    return discriminator_loss, discriminator_accuracy.result()

### Generator のトレーニング ステップ

同様に、次のステップを含む Generator 固有のトレーニング ステップを `generator_train_step` 関数で定義します。
1. 潜在空間内のランダムな点をサンプリングします。
2. 生成された画像（1）に対して、本物の画像であるかのようにラベルを作成します。
3. `tf.GradientTape` コンテキスト内で Generator を呼び出して画像を生成し、それらを Discriminator に渡して、ステップ 2 で作成したラベルを使用して損失を計算します。
4. `GradientTape` から Generator のパラメータの勾配を取得し、それらを更新します。Discriminator は Generator のトレーニング ステップでは更新されないことに注意してください。

**演習**: 上記の手順および Discriminator のトレーニング プロセスに従って、Generator のトレーニング ステップを定義してください。

In [ ]:
def generator_train_step(
    discriminator,
    generator,
    generator_optimizer,
    loss_fn,
    batch_size,
    input_digits,
):
    # 潜在空間内のランダムな点をサンプリングする。
    random_latent_vectors = keras.random.normal(shape=(batch_size, LATENT_DIM))

    # 本物の画像であるかのようにラベルを作成する。
    labels = keras.ops.ones(shape=(batch_size, 1))

    # TODO: トレーニング プロセスを完成させる。
    # Generator と Discriminator を呼び出し、Generator の損失を計算する。
    with tf.GradientTape() as tape:
        ...

    # TODO: Generator の重みの勾配を取得する。
    # Generator のみをトレーニングする。
    grads = tape.gradient(...)

    generator_optimizer.apply_gradients(
        grads_and_vars=zip(grads, generator.trainable_weights)
    )

    return generator_loss

### 外側のトレーニング ループ
これで両方のトレーニング ループが定義されました。<br>
それらを `train_step` 関数内にラップし、外側のトレーニング ループを定義しましょう。

ここでは、Discriminator と Generator を順番にトレーニングするために `discriminator_train_step` と `generator_train_step` を呼び出し、2 つのモデルの 2 つの損失を返します。

In [ ]:
DISCRIMINATOR_OPTIMIZER = keras.optimizers.Adam(
    learning_rate=0.0002, beta_1=0.5, beta_2=0.999, epsilon=1e-8
)
GENERATOR_OPTIMIZER = keras.optimizers.Adam(
    learning_rate=0.0004, beta_1=0.5, beta_2=0.999, epsilon=1e-8
)

LOSS_FN = keras.losses.BinaryCrossentropy(from_logits=True)

ONE_SIDED_LABEL_SMOOTHING = 0.9

DISCRIMINATOR_ACCURACY = keras.metrics.BinaryAccuracy()


def train_step(features):
    real_images = features["images"]
    input_digits = features["digits"]

    # ランダム ノイズ ベクトルに使用する動的なバッチサイズを取得。
    batch_size = keras.ops.shape(real_images)[0]

    # まず Discriminator をトレーニング。
    discriminator_loss, discriminator_accuracy_value = discriminator_train_step(
        dcgan_discriminator,
        dcgan_generator,
        DISCRIMINATOR_OPTIMIZER,
        LOSS_FN,
        DISCRIMINATOR_ACCURACY,
        ONE_SIDED_LABEL_SMOOTHING,
        batch_size,
        real_images,
        input_digits,
    )

    # 次に Generator をトレーニング。
    generator_loss = generator_train_step(
        dcgan_discriminator,
        dcgan_generator,
        GENERATOR_OPTIMIZER,
        LOSS_FN,
        batch_size,
        input_digits,
    )
    losses = {
        "discriminator_loss": discriminator_loss,
        "discriminator_accuracy": discriminator_accuracy_value,
        "generator_loss": generator_loss,
    }

    return losses

### tf.data による入力データ パイプラインの作成
モデルの準備ができました。tf.data を使用してデータ パイプラインを作成しましょう。<br>
cGAN には画像だけでなくラベルも渡す必要があることに注意してください。

In [ ]:
def create_train_dataset(batch_size):
    # MNIST データを取得してフォーマット。
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

    # トレーニング セットとテスト セットの両方を結合。
    combined_images = np.concatenate([x_train, x_test])
    combined_images = (combined_images.astype("float32") - 127.5) / 255.0
    combined_images = np.reshape(combined_images, newshape=(-1, 28, 28, 1))

    combined_labels = np.concatenate([y_train, y_test])

    # トレーニング用の tf.data.Dataset を作成。
    dataset = tf.data.Dataset.from_tensor_slices(
        tensors={"images": combined_images, "digits": combined_labels}
    )
    dataset = dataset.shuffle(buffer_size=10000).batch(
        batch_size=batch_size, drop_remainder=True
    )

    return dataset

In [ ]:
BATCH_SIZE = 128
EPOCHS = 10  # より良い生成結果を得るには数値を変更してください

training_dataset = create_train_dataset(batch_size=BATCH_SIZE)

dataset_length = training_dataset.cardinality().numpy()

## CGAN のトレーニング
ループ内で `train_step` 関数を呼び出すことで、モデルをトレーニングできます。<br>
GPU（NVIDIA T4）を使用した場合、10 エポックのトレーニングには約 7 分かかります。

In [ ]:
output_keys = ["discriminator_loss", "discriminator_accuracy", "generator_loss"]

history = {key: [] for key in output_keys}

for epoch in range(EPOCHS):
    print(f"{'=' * 20} EPOCH {epoch + 1} {'=' * 20}")
    epoch_history = {key: [] for key in output_keys}

    for i, features in enumerate(training_dataset, 1):
        losses = train_step(features)
        for key in output_keys:
            epoch_history[key].append(losses[key])

        if i % 100 == 0 or i == dataset_length:
            performance_str = ", ".join(
                [
                    f"{key}: {keras.ops.mean(epoch_history[key]):5f}"
                    for key in output_keys
                ]
            )
            print(f"step {i:3d} / {dataset_length}: {performance_str}")

    for key in output_keys:
        history[key].append(keras.ops.mean(epoch_history[key]))

    DISCRIMINATOR_ACCURACY.reset_state()

トレーニング済みモデルを使用して、損失の履歴と生成されたいくつかの画像をプロットしてみましょう。

In [ ]:
def plot_loss_history(history):
    plt.plot(history["discriminator_loss"])
    plt.plot(history["discriminator_accuracy"])
    plt.plot(history["generator_loss"])
    plt.title("Training losses")
    plt.ylabel("Loss")
    plt.xlabel("Epoch")
    plt.legend(output_keys, loc="upper left")
    plt.show()

In [ ]:
plot_loss_history(history=history)

損失曲線はどのように見えますか？

Discriminator と Generator は異なる損失値を最適化しようとしており、互いに競い合っていることを思い出してください。<br>
したがって、トレーニング中に一方の損失が低下し、もう一方が急激に上昇している場合、2 つのモデル間のバランスに問題があることが分かります。その場合は、この 2 つのバランスを取るために最適化戦略、学習率スケジューラ、またはモデル アーキテクチャ自体の調整を試みる必要があります。

また、Discriminator の精度（accuracy）を監視することも有用です。<br>
それが 50% に近い場合、Discriminator が本物か偽物かを分類できなくなっていることを意味します。これにより、Generator が本物のような画像を作成する非常に優れた仕事をしているか、Discriminator がまったく適合していないかのいずれかであることが分かります。

## 画像の生成
これで cGAN は画像を生成する準備が整いました。Generator に生成させたい数字を指定できることに注目してください。

In [ ]:
LABEL_TO_GENERATE = 9  # 0-9
latent_vectors = keras.random.normal(shape=(1, LATENT_DIM))

generated_image = dcgan_generator(
    inputs=[
        latent_vectors,
        keras.ops.convert_to_tensor([LABEL_TO_GENERATE], dtype="int32"),
    ]
)
plt.imshow(
    keras.ops.reshape(generated_image, generated_image.shape[1:-1]), cmap="gray"
)

複数の数字を生成してみましょう。

これは複数の画像を可視化するためのヘルパー関数です。

In [ ]:
def plot_images(images):
    """画像をプロットします。

    Args:
        images: np.array、[num_images, image_size, image_size, num_channels]
            の画像の配列。
    """
    num_images = len(images)

    plt.figure(figsize=(20, 20))
    for i in range(num_images):
        image = images[i]
        plt.subplot(1, num_images, i + 1)
        plt.xticks([])
        plt.yticks([])
        plt.grid(False)
        plt.imshow(keras.ops.reshape(image, image.shape[:-1]), cmap="gray")
    plt.show()

In [ ]:
num_examples = 20
latent_vectors = keras.random.normal(shape=(num_examples, LATENT_DIM))
for i in range(10):
    labels = keras.ops.ones(shape=num_examples, dtype="int32") * i
    plot_images(images=dcgan_generator(inputs=[latent_vectors, labels]))

良さそうです！エポック数を大きく指定（50 など）することで、より高品質な画像を得ることができます。

## （オプション）GAN クラスの定義
別々のトレーニング ループを定義する代わりに、[keras.Model](https://keras.io/api/models/model/) をサブクラス化してクラスにまとめることもできます。

独自のモデル クラスを定義することで、元のトレーニング ステップを簡単に定義して `model.fit()` などの標準モデル API を使用できるようになります。また、keras.Model の関数をオーバーライドしてモデルの動作を変更することもできます。

上記で定義したトレーニング関数に加えて、ここでは Conditional_GAN クラスに次の関数を追加します。
- `__init__()`: モデル クラスをインスタンス化します。ここで、上で定義した Generator と Discriminator を渡します。また、Discriminator のトレーニングで使用される `one_sided_label_smoothing` 定数もここで指定することに注意してください。
- `compile()`: `Model.compile()` 関数をオーバーライドして、Generator と Discriminator のオプティマイザを別々に定義できるようにし、両方のモデルで使用される損失関数を追加します。
- `metrics()`: モデルのメトリックを定義します。この `metrics` プロパティにメトリックを追加することで、各エポックの開始時や `evaluate()` の開始時に `reset_states()` が自動的に呼び出されるようにします。

In [ ]:
class Conditional_GAN(keras.Model):
    def __init__(
        self, discriminator, generator, latent_dim, one_sided_label_smoothing
    ):
        super().__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        self.one_sided_label_smoothing = one_sided_label_smoothing

    def compile(
        self,
        discriminator_optimizer,
        generator_optimizer,
        loss_fn,
        discriminator_accuracy,
    ):
        super().compile()
        self.discriminator_optimizer = discriminator_optimizer
        self.generator_optimizer = generator_optimizer
        self.loss_fn = loss_fn
        self.discriminator_accuracy = discriminator_accuracy

    def discriminator_train_step(self, batch_size, real_images, input_digits):
        # 上で定義した関数を再利用
        return discriminator_train_step(
            self.discriminator,
            self.generator,
            self.discriminator_optimizer,
            self.loss_fn,
            self.discriminator_accuracy,
            self.one_sided_label_smoothing,
            batch_size,
            real_images,
            input_digits,
        )

    def generator_train_step(self, batch_size, input_digits):
        # 上で定義した関数を再利用
        return generator_train_step(
            self.discriminator,
            self.generator,
            self.generator_optimizer,
            self.loss_fn,
            batch_size,
            input_digits,
        )

    def train_step(self, features):
        real_images = features["images"]
        input_digits = features["digits"]

        # ランダム ノイズ ベクトルに使用する動的なバッチサイズを取得。
        batch_size = keras.ops.shape(real_images)[0]

        # まず Discriminator をトレーニング。
        (
            discriminator_loss,
            discriminator_accuracy_value,
        ) = self.discriminator_train_step(batch_size, real_images, input_digits)

        # 次に Generator をトレーニング。
        generator_loss = self.generator_train_step(batch_size, input_digits)
        losses = {
            "discriminator_loss": discriminator_loss,
            "discriminator_accuracy": discriminator_accuracy_value,
            "generator_loss": generator_loss,
        }
        return losses

    @property
    def metrics(self):
        return [self.discriminator_accuracy]

Conditional GAN モデルをインスタンス化してコンパイルできます。

インスタンス化によって `__init__` 関数が呼び出され、コンパイルによって `Conditional_GAN` クラスの `compile` 関数が呼び出されることに注意してください。

In [ ]:
dcgan_generator = create_dcgan_generator(
    latent_dim=LATENT_DIM,
    num_classes=NUM_CLASSES,
    projection_shape=PROJECTION_SHAPE,
)

dcgan_discriminator = create_dcgan_discriminator(
    image_shape=IMAGE_SHAPE, num_classes=NUM_CLASSES
)

# DCGAN の Generator と Discriminator を使用して Conditional_GAN インスタンスをインスタンス化。
dcgan = Conditional_GAN(
    discriminator=dcgan_discriminator,
    generator=dcgan_generator,
    latent_dim=LATENT_DIM,
    one_sided_label_smoothing=0.9,
)

dcgan.compile(
    discriminator_optimizer=keras.optimizers.Adam(
        learning_rate=0.0002, beta_1=0.5, beta_2=0.999, epsilon=1e-8
    ),
    generator_optimizer=keras.optimizers.Adam(
        learning_rate=0.0004, beta_1=0.5, beta_2=0.999, epsilon=1e-8
    ),
    loss_fn=keras.losses.BinaryCrossentropy(from_logits=True),
    discriminator_accuracy=keras.metrics.BinaryAccuracy(),
)

## CGAN のトレーニング
`model.fit()` を呼び出すだけでモデルをトレーニングできます。<br>
GPU（NVIDIA T4）を使用した場合、10 エポックのトレーニングには約 7 分かかります。

In [ ]:
BATCH_SIZE = 128
EPOCHS = 10  # より良い生成結果を得るには数値を変更してください

# DCGAN モデルをトレーニング。
dcgan_history = dcgan.fit(
    create_train_dataset(batch_size=BATCH_SIZE), epochs=EPOCHS
)

In [ ]:
plot_loss_history(history=dcgan_history.history)

In [ ]:
num_examples = 20
latent_vectors = keras.random.normal(shape=(num_examples, LATENT_DIM))
for i in range(10):
    labels = keras.ops.ones(shape=num_examples, dtype="int32") * i
    plot_images(images=dcgan_generator(inputs=[latent_vectors, labels]))

## まとめ
このノートブックでは、Conditional GAN モデルの構築とトレーニングの方法を学びました。

Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License